# Stage 3: Data Preprocessing

## Objective

The objective of data preprocessing is to transform the raw
Bank Marketing dataset into a suitable format for machine
learning classification algorithms.

### Preprocessing Activities

1. Remove target leakage.
2. Separate independent and dependent variables.
3. Encode the target variable.
4. Split the dataset into training and testing sets.
5. Encode categorical features.
6. Scale numerical features.
7. Validate the processed dataset.

The original dataset will remain unchanged.

In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)
from sklearn.compose import ColumnTransformer

df = pd.read_csv("../data/raw/bank.csv", sep=";")

print("Dataset Shape:", df.shape)

Dataset Shape: (4521, 17)


## 3.1 Remove Target Leakage

The feature `duration` represents the duration of the last
customer interaction.

This information is available only after the interaction
has taken place.

Including it in the model could result in target leakage.

Therefore, duration will be excluded from the independent
variables used for model training.

The original dataset remains unchanged.

In [4]:
df_model = df.drop(columns=["duration"])

print("Original Dataset Shape:", df.shape)
print("Model Dataset Shape:", df_model.shape)

df_model.head()

Original Dataset Shape: (4521, 17)
Model Dataset Shape: (4521, 16)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,1,-1,0,unknown,no


## 3.2 Feature and Target Separation

Independent Variables (X):
All input features used to predict customer subscription.

Dependent Variable (y):
The target variable representing subscription status.

Target:
- yes: Customer subscribed.
- no: Customer did not subscribe.

In [6]:
X = df_model.drop(columns=["y"])
y = df_model["y"]

print("Independent Variables Shape:", X.shape)
print("Dependent Variable Shape:", y.shape)

Independent Variables Shape: (4521, 15)
Dependent Variable Shape: (4521,)


## 3.3 Target Variable Encoding

Machine learning algorithms require numerical target labels.

We will use LabelEncoder to transform the target classes.

Mapping:
- no  -> 0
- yes -> 1

In [8]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Target Mapping:")

for label, encoded_value in zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
):
    print(f"{label} -> {encoded_value}")

Target Mapping:
no -> 0
yes -> 1


In [9]:
print(pd.Series(y_encoded).value_counts())

0    4000
1     521
Name: count, dtype: int64


## 3.4 Train-Test Split

The dataset will be divided into two subsets.

Training Dataset:
Used to train machine learning models.

Testing Dataset:
Used to evaluate model performance on unseen observations.

Split Ratio:
- Training: 80%
- Testing: 20%

Stratified Sampling:
The stratify parameter maintains the original target
class distribution in both training and testing datasets.

Random State:
42, to ensure reproducibility.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (3616, 15)
Testing Features: (905, 15)
Training Target: (3616,)
Testing Target: (905,)


In [12]:
print("Training Distribution:")
print(pd.Series(y_train).value_counts(normalize=True) * 100)

print("\nTesting Distribution:")
print(pd.Series(y_test).value_counts(normalize=True) * 100)

Training Distribution:
0    88.46792
1    11.53208
Name: proportion, dtype: float64

Testing Distribution:
0    88.508287
1    11.491713
Name: proportion, dtype: float64


In [13]:
print("Training Distribution:")
print(pd.Series(y_train).value_counts(normalize=True) * 100)

print("\nTesting Distribution:")
print(pd.Series(y_test).value_counts(normalize=True) * 100)

Training Distribution:
0    88.46792
1    11.53208
Name: proportion, dtype: float64

Testing Distribution:
0    88.508287
1    11.491713
Name: proportion, dtype: float64


## 3.5 Feature Type Identification

The independent variables contain numerical and categorical features.

Numerical Features:
- age
- balance
- day
- campaign
- pdays
- previous

Categorical Features:
- job
- marital
- education
- default
- housing
- loan
- contact
- month
- poutcome

Different preprocessing techniques will be applied
to each feature type.

In [14]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:", numerical_features)
print("\nCategorical Features:", categorical_features)

Numerical Features: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

Categorical Features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 3.6 Feature Transformation

### A. One-Hot Encoding

Categorical variables contain textual values.

One-Hot Encoding converts each category into a separate
binary feature.

Example:

job = management

Possible encoded representation:

job_management = 1
job_services = 0
job_technician = 0

We will use OneHotEncoder with handle_unknown="ignore"
to handle categories not encountered during training.

### B. Standardization

Numerical features have different ranges.

StandardScaler transforms numerical features using:

z = (x - mean) / standard_deviation

After standardization:
- Mean approximately equals 0.
- Standard deviation approximately equals 1.

This is particularly important for Logistic Regression,
KNN and SVM.

In [17]:
numerical_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 3.7 Apply Feature Transformations

The preprocessing pipeline will be fitted using training data only.

Training Data:
- Fit StandardScaler.
- Learn categorical encoding.
- Transform training features.

Testing Data:
- Reuse the transformations learned from training data.
- Transform testing features.

This prevents data leakage from the testing dataset.

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

Processed Training Shape: (3616, 50)
Processed Testing Shape: (905, 50)


In [19]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

display(X_train_processed_df.head())

,num__age,num__balance,num__day,num__campaign,num__pdays,num__previous,cat__job_admin.,cat__job_blue-collar,cat__job_entrepreneur,cat__job_housemaid,...,cat__month_jun,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep,cat__poutcome_failure,cat__poutcome_other,cat__poutcome_success,cat__poutcome_unknown
0,-0.864347,-0.505865,-0.492444,0.065082,-0.411541,-0.329453,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-0.673204,-0.448232,-0.248716,-0.570967,0.447116,11.136987,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,0.282513,-0.499461,0.726196,-0.252942,-0.411541,-0.329453,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.673204,2.456534,0.360604,-0.570967,-0.411541,-0.329453,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.951514,1.297118,-0.004988,1.337181,-0.411541,-0.329453,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 3.8 Save Preprocessing Artifacts

Save the fitted preprocessing object and processed datasets.

The fitted preprocessor will be reused during:
- Model development.
- Model evaluation.
- Streamlit application deployment.

Saving the fitted transformer ensures that new customer
data receives exactly the same transformations.

In [20]:
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Save fitted preprocessing object
joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

# Save target encoder
joblib.dump(
    label_encoder,
    "../models/label_encoder.joblib"
)

# Save processed training dataset
train_data = X_train_processed_df.copy()
train_data["target"] = y_train

train_data.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

# Save processed testing dataset
test_data = X_test_processed_df.copy()
test_data["target"] = y_test

test_data.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Preprocessing artifacts saved successfully!")

Preprocessing artifacts saved successfully!
